# 第1部分：数据加载

**目的：** 从阿里云ODPS数据仓库中读取贷款申请数据

**ODPS是什么？** 阿里云的大数据平台，类似于一个超大的数据库，存放着公司所有的业务数据表

In [ ]:
from odps.df import DataFrame
import multiprocessing
from datetime import datetime

# 连接ODPS数据源
# data_id 是在机器学习平台上配置好的数据源ID，每个项目不同
odps = mlp.get_odps_instance(data_id="data13a88922e07511f089a70242ac850002")

## 读取数据函数

为什么用多进程？
- 数据量很大（几万~几十万行 × 几百列）
- 单进程读取太慢，多进程可以利用所有CPU核心并行读取

In [ ]:
def read_data_from_odps(odpstablename):
    """
    从ODPS读取一张表的数据，返回pandas DataFrame
    
    参数：
        odpstablename: ODPS中的表名（字符串）
    返回：
        df: pandas的DataFrame，可以像Excel表格一样操作
    """
    # 获取表对象
    table = DataFrame(odps.get_table(odpstablename))
    
    # 自动检测CPU核数，用所有核心来加速读取
    n_process = multiprocessing.cpu_count()
    print('使用CPU核数:', n_process)
    
    # 记录开始时间
    dt1 = datetime.now()
    print('开始读取:', dt1.strftime("%Y-%m-%d %H:%M:%S"))
    
    # 执行读取（这步可能要几分钟，取决于数据量）
    df = table.to_pandas(n_process=n_process)
    
    # 记录结束时间，打印耗时
    dt2 = datetime.now()
    duration = dt2 - dt1
    print(f'读取完成，耗时: {duration.seconds/60:.1f}分钟，数据形状: {df.shape}')
    # df.shape 返回 (行数, 列数)
    # 比如 (50000, 300) 表示5万条数据、300个特征
    
    return df

## 实际使用

表名解读：`yy_apply_kb_zj_05_nobr_xf_pass_rule_01_202506_0617`
- yy = 营运
- apply = 申请
- kb = 客群
- zj = 朱静（你的标识）
- nobr = 无百融（不含百融数据源）
- xf = 消费
- pass_rule = 通过规则的
- 202506_0617 = 2025年6月数据，6月17日跑的

In [ ]:
df = read_data_from_odps('yy_apply_kb_zj_05_nobr_xf_pass_rule_01_202506_0617')

# 查看前3行，确认读取正确
df.head(3)

---
### 面试考点

| 问题 | 答案 |
|------|------|
| 为什么用多进程？ | 数据量大，单进程IO瓶颈，多进程并行读取提速 |
| 宽表是什么？ | 一行=一个客户，列=所有特征（可能300+列），已经做好了join |
| 为什么打印时间？ | 方便排查性能问题，知道读取瓶颈在哪里 |